In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

ARTIFACTS = Path("../artifacts")

df = pd.read_csv(ARTIFACTS / "clean_data.csv")
TARGET = "CO2 uptake (mmol/g)"
FEATURES = [c for c in df.columns if c not in ["row_id", TARGET, "split"]]

final_models = joblib.load(ARTIFACTS / "final_models.joblib")
with open(ARTIFACTS / "ensemble_weights.json") as f:
    weights = json.load(f)

In [2]:
def predict_uptake(sample: dict) -> float:
    X = pd.DataFrame([sample], columns=FEATURES)
    return float(sum(w * final_models[name].predict(X)[0] for name, w in weights.items()))

In [3]:
test = df[df["split"] == "test"]
row = test.iloc[0]
sample = row[FEATURES].to_dict()

print("Actual:   ", round(row[TARGET], 3))
print("Predicted:", round(predict_uptake(sample), 3))

sample_missing = dict(sample)
sample_missing.pop("Micropore Volume (cm3/g)")
print("Predicted with a blank input:", round(predict_uptake(sample_missing), 3))

Actual:    2.625
Predicted: 2.908
Predicted with a blank input: 3.521


In [4]:
feature_info = {
    f: {"min": float(df[f].min()), "max": float(df[f].max()), "median": float(df[f].median())}
    for f in FEATURES
}
with open(ARTIFACTS / "feature_info.json", "w") as f:
    json.dump(feature_info, f, indent=2)
pd.DataFrame(feature_info).T.round(2)

,min,max,median
Surface Area (m2/g),2.48,3337.00,1420.00
Total Pore Volume(cm3/g),0.02,3.09,0.68
Micropore Volume (cm3/g),0.04,1.29,0.48
C (%),40.40,96.96,79.10
H (%),0.00,6.01,1.50
N (%),0.00,38.15,2.15
O (%),0.00,54.00,15.42
S (%),0.00,13.20,0.00
Temp (°C),0.00,30.00,25.00
Pressure (bar),0.10,1.01,1.00


In [5]:
import sklearn, xgboost, lightgbm, catboost

bundle = {"models": final_models, "weights": weights,
          "features": FEATURES, "target": TARGET}
joblib.dump(bundle, ARTIFACTS / "model_bundle.joblib")

for lib in [sklearn, xgboost, lightgbm, catboost, joblib, pd, np]:
    print(lib.__name__, lib.__version__)

sklearn 1.7.2
xgboost 3.4.1
lightgbm 4.7.0
catboost 1.2.10
joblib 1.5.2
pandas 2.3.3
numpy 2.3.5
